# Lecture 8: Serving LLMs and Embedding Models

This notebook is a tutorial for production-oriented NLP serving.

## Learning outcomes
1. Compare proprietary and open-source serving stacks.
2. Use Hugging Face chat templates correctly for instruct models.
3. Understand vLLM inference methodology and key inference arguments.
4. Serve both generation and embedding workloads with practical patterns.

## 1) Serving landscape

### Proprietary model serving
- API gateways and managed providers: OpenRouter, Amazon Bedrock, Azure OpenAI, Google Vertex AI.
- Strengths: fast onboarding, managed reliability, model variety, less ops burden.
- Tradeoff: lower infra control, provider-specific limits/pricing, data-governance constraints.

### Open-source model serving
- Self-host or managed OSS stacks: Hugging Face (Transformers/TGI/Inference Endpoints), vLLM.
- Strengths: full control over model/runtime/caching, lower cost at scale, customizable latency-quality tradeoffs.
- Tradeoff: GPU ops, observability, autoscaling, model lifecycle ownership.

### Quick decision guide
- Need fastest prototype and wide model catalog: start with OpenRouter/Bedrock.
- Need custom prompts/templates/runtime tuning or strict hosting control: use Hugging Face + vLLM.

## 2) Proprietary serving options

### OpenRouter
- Single API interface across many model providers.
- Useful for rapid A/B testing across models with minimal client-code changes.

### Amazon Bedrock
- AWS-native managed access to foundation models with IAM, VPC, CloudWatch integration.
- Strong fit for organizations already in AWS and requiring enterprise controls.

### Typical enterprise pattern
- API Gateway + auth + rate limits
- Prompt guardrails and PII filters
- Centralized logs/metrics/traces
- Fallback model routing for availability

In [ ]:
# Optional setup if you want to run all examples in this notebook
%pip install -q transformers vllm sentence-transformers requests boto3

In [ ]:
# OpenRouter text generation example
import os
import requests

OPENROUTER_API_KEY = "sk-or-v1-52cd0c37b64fa5ce01f0758**********931e837afb574ae88498090"

url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "model": "meta-llama/llama-3.1-8b-instruct",
    "messages": [
        {"role": "system", "content": "You are a concise NLP tutor."},
        {"role": "user", "content": "Explain embeddings in 3 bullets."}
    ],
    "temperature": 0.2 # details
}

# response = requests.post(url, headers=headers, json=payload, timeout=60)
# print(response.json())

In [2]:
response = requests.post(url, headers=headers, json=payload, timeout=60)
print(response.json())

{'id': 'gen-1772591541-eMdhVsQsWL2YtSrAfvHz', 'object': 'chat.completion', 'created': 1772591541, 'model': 'meta-llama/llama-3.1-8b-instruct', 'provider': 'Groq', 'system_fingerprint': 'fp_4387d3edbb', 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': 'Here are 3 key points about embeddings:\n\n• **Definition**: Embeddings are a way to represent words or other data points as dense vectors in a high-dimensional space, allowing for semantic relationships and patterns to be captured. This is often achieved through techniques like Word2Vec or GloVe.\n\n• **Properties**: Embeddings preserve the semantic meaning of words, enabling tasks like word analogy (e.g., "king" - "man" + "woman" = "queen") and word similarity (e.g., "dog" and "cat" are similar). They also reduce the dimensionality of the input data, making it more efficient for processing.\n\n• **Applications**: Embeddings are widely used in 

In [ ]:
"""
Lists the available Amazon Bedrock models in an &AWS-Region;.
"""
import logging
import json
import boto3


from botocore.exceptions import ClientError


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def list_foundation_models(bedrock_client):
    """
    Gets a list of available Amazon Bedrock foundation models.

    :return: The list of available bedrock foundation models.
    """

    try:
        response = bedrock_client.list_foundation_models()
        models = response["modelSummaries"]
        logger.info("Got %s foundation models.", len(models))
        return models

    except ClientError:
        logger.error("Couldn't list foundation models.")
        raise


def main():
    """Entry point for the example. Change aws_region to the &AWS-Region;
    that you want to use."""
   
    aws_region = "us-east-1"

    bedrock_client = boto3.client(service_name="bedrock", region_name=aws_region)
    
    fm_models = list_foundation_models(bedrock_client)
    for model in fm_models:
        print(f"Model: {model["modelName"]}")
        print(json.dumps(model, indent=2))
        print("---------------------------\n")
    
    logger.info("Done.")

if __name__ == "__main__":
    main()


In [ ]:
# Use the Conversation API to send a text message to Amazon Titan Text G1 - Express.

import boto3
from botocore.exceptions import ClientError

# Create an Amazon Bedrock Runtime client.
brt = boto3.client("bedrock-runtime")

# Set the model ID, e.g., Amazon Titan Text G1 - Express.
model_id = "amazon.titan-text-express-v1"

# Start a conversation with the user message.
user_message = "Describe the purpose of a 'hello world' program in one line."
conversation = [
    {
        "role": "user",
        "content": [{"text": user_message}],
    }
]

try:
    # Send the message to the model, using a basic inference configuration.
    response = brt.converse(
        modelId=model_id,
        messages=conversation,
        inferenceConfig={"maxTokens": 512, "temperature": 0.5, "topP": 0.9},
    )

    # Extract and print the response text.
    response_text = response["output"]["message"]["content"][0]["text"]
    print(response_text)

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
    exit(1)

## 3) Open-source serving: Hugging Face and vLLM

We now go deeper into Hugging Face (prompt formatting and embeddings) and vLLM (inference runtime and tuning).

## 4) Hugging Face deep dive: chat templates

For instruct-tuned chat models, formatting is critical. Different models expect different special tokens.

`tokenizer.apply_chat_template(...)` converts role-based messages into the exact prompt string/token IDs expected by the model.

Important flags:
- `tokenize=False`: returns rendered text prompt (useful for debugging).
- `tokenize=True`: returns token IDs directly.
- `add_generation_prompt=True`: appends the assistant prefix so generation starts in the right place.
- Batch mode: pass `List[List[message_dict]]` to template multiple chats at once.

In [3]:
from transformers import AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

messages = [
    {"role": "system", "content": "You are a helpful NLP teaching assistant."},
    {"role": "user", "content": "Explain what semantic similarity means."},
]

rendered = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(rendered[:500])

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)
print("Tokenized shape:", tuple(input_ids.shape))

/Users/gauravmapari/COEP/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<|im_start|>system
You are a helpful NLP teaching assistant.<|im_end|>
<|im_start|>user
Explain what semantic similarity means.<|im_end|>
<|im_start|>assistant



AttributeError: 

In [ ]:
# Batch chat-template usage
batch_messages = [
    [
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Define perplexity."},
    ],
    [
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Define cross entropy."},
    ],
]

batch_prompts = tokenizer.apply_chat_template(
    batch_messages,
    tokenize=False,
    add_generation_prompt=True,
)

for i, p in enumerate(batch_prompts):
    print(f"Prompt {i} length: {len(p)}")

# Common pitfall: If add_generation_prompt=False, some models may stop after user turn
# because assistant prefix tokens are missing.

## 5) Hugging Face for embeddings

Common choices:
- `sentence-transformers` for easy embedding inference in Python services.
- Hugging Face Inference Endpoints for managed hosting.
- Text Embeddings Inference (TEI) for high-throughput embedding servers.

Design tips:
- Normalize vectors when cosine similarity is your retrieval metric.
- Cache embeddings for repeated documents/queries.
- Keep separate models for document vs query embeddings when recommended by model docs.

In [ ]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5") #LM v6
texts = [
    "Transformers use self-attention.",
    "Embeddings map text to dense vectors.",
]
embeddings = emb_model.encode(texts, normalize_embeddings=True)
print("Embedding matrix shape:", embeddings.shape)

## 6) vLLM deep dive: inference methodology and arguments

### Methodology (practical sequence)
1. Initialize `LLM(...)` with memory/throughput-aware engine arguments.
2. Convert role messages to model-specific prompts (chat templates).
3. Configure `SamplingParams` (temperature, max tokens, guided decoding, etc.).
4. Generate with batch requests to leverage continuous batching and KV-cache efficiency.
5. Parse outputs and post-validate constrained formats (for example JSON).

### High-impact inference arguments
- `gpu_memory_utilization`: controls how aggressively vLLM uses GPU memory.
- `max_model_len`: max context length; larger values increase KV-cache pressure.
- `max_num_batched_tokens`: throughput control for token-level batching.
- `max_num_seqs`: cap concurrent sequences in scheduler.
- `dtype`: precision choice (`bfloat16`, `float16`, etc.) affecting speed/memory.
- `compilation_config`: CUDA graph / compilation tuning for steady-state latency.
- `disable_log_stats`: reduces runtime logging overhead in hot paths.

In [ ]:
#Metaflow

# Open source platform for all datascience training/inference workflows.

# Developed by Netflix

Nvidia L40S compute pools on AWS #3 dollars


In [ ]:
# Adapted from your sample: reusable self-hosted inference wrapper
from typing import Any, Dict, List, Optional
import json


class SelfHostedAIInfer:
    def __init__(self, model_path: str, engine_args: Optional[Dict[str, Any]] = None):
        from transformers import AutoTokenizer
        from vllm import LLM
        from vllm.config import CompilationConfig

        self.model_path = model_path

        # Default optimized configuration for L40S + Gemma 3 12B
        default_args = {
            "model": model_path,
            "disable_log_stats": True,
            "gpu_memory_utilization": 0.95,
            "max_model_len": 2048,
            "dtype": "bfloat16",
            "max_num_batched_tokens": 16384,
            "max_num_seqs": 128,
            "compilation_config": CompilationConfig(
                level=3,
                cudagraph_capture_sizes=[32, 64, 128, 256],
                cudagraph_num_of_warmups=1,
                use_cudagraph=True,
            ),
        }

        final_args = {**default_args, **engine_args} if engine_args else default_args

        self.llm = LLM(**final_args)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)

    def _tokenize(self, prompts: List[str], system_prompt: Optional[str]) -> List[str]:
        messages_list: List[List[Dict[str, str]]] = []
        for prompt in prompts:
            messages: List[Dict[str, str]] = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": prompt})
            messages_list.append(messages)

        templated = self.tokenizer.apply_chat_template(
            messages_list, tokenize=False, add_generation_prompt=True
        )
        return templated if isinstance(templated, list) else [templated]

    def _generate(
        self,
        texts: List[str],
        temperature: float,
        json_schema: Optional[Dict[str, Any]],
    ):
        from vllm import SamplingParams
        from vllm.sampling_params import GuidedDecodingParams

        if json_schema:
            #model to output Structured JSON
            guided_decoding_params_json = GuidedDecodingParams(json=json_schema)
            sampling_params = SamplingParams(
                temperature=temperature,
                max_tokens=2048,
                guided_decoding=guided_decoding_params_json,
            )
        else:
            sampling_params = SamplingParams(temperature=temperature, max_tokens=2048)

        # Optional warmup with first prompt to stabilize first-token latency
        if texts:
            _ = self.llm.generate(texts[0], sampling_params)

        return self.llm.generate(texts, sampling_params, use_tqdm=True)

    def _parse_outputs(self, outputs) -> List[Dict[str, Any]]:
        parsed_result: List[Dict[str, Any]] = []
        for output in outputs:
            parsed_result.append(
                {
                    "prompt_processed": output.prompt,
                    "generated_text": output.outputs[0].text if output.outputs else "",
                }
            )
        return parsed_result

    def _validate_json_output(self, output: str) -> str:
        output = output.strip()

        try:
            json.loads(output)
            return output
        except json.JSONDecodeError:
            pass

        if "```json" in output:
            start = output.find("```json") + 7
            end = output.find("```", start)
            if end != -1:
                extracted = output[start:end].strip()
                try:
                    json.loads(extracted)
                    return extracted
                except json.JSONDecodeError:
                    pass

        start = output.find("{")
        end = output.rfind("}")
        if start != -1 and end != -1 and end > start:
            extracted = output[start : end + 1]
            try:
                json.loads(extracted)
                return extracted
            except json.JSONDecodeError:
                pass

        return output

    def infer(
        self,
        prompts: List[str],
        system_prompt: Optional[str] = None,
        temperature: float = 0.0,
        json_schema: Optional[Dict[str, Any]] = None,
    ) -> List[str]:
        texts = self._tokenize(prompts, system_prompt)
        outputs = self._generate(texts, temperature, json_schema)
        parsed_result = self._parse_outputs(outputs)
        output_texts = [o["generated_text"] for o in parsed_result]

        if json_schema:
            output_texts = [self._validate_json_output(p) for p in output_texts]

        return output_texts

In [ ]:
# Example usage: normal chat generation
# infer_engine = SelfHostedAIInfer(model_path="google/gemma-3-12b-it")
# results = infer_engine.infer(
#     prompts=["Explain BM25 vs dense retrieval.", "What is reranking?"],
#     system_prompt="You are an NLP professor.",
#     temperature=0.2,
# )
# print(results[0])

# Example usage: guided JSON output
# schema = {
#     "type": "object",
#     "properties": {
#         "topic": {"type": "string"},
#         "summary": {"type": "string"},
#     },
#     "required": ["topic", "summary"],
# }
# json_results = infer_engine.infer(
#     prompts=["Summarize transformer inference optimization."],
#     system_prompt="Return valid JSON only.",
#     json_schema=schema,
# )
# print(json_results[0])

## 7) vLLM service mode (OpenAI-compatible API)

You can serve models behind an HTTP API and use standard OpenAI-compatible clients.

```bash
# Text generation server
vllm serve Qwen/Qwen2.5-7B-Instruct --dtype bfloat16 --max-model-len 4096

# Embedding server (task flag can differ by vLLM version; check --help)
vllm serve BAAI/bge-large-en-v1.5 --task embed --dtype float16
```

```bash
curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "Explain tokenization."}]
  }'
```

Operational notes:
- Track tokens/sec, queue wait, and p95 latency separately.
- Keep prompts batched whenever possible.
- Use model-specific context limits and avoid unnecessary long prompts.

## 8) Class exercise

1. Pick one proprietary route (OpenRouter or Bedrock) and one open-source route (vLLM).
2. Run the same 5 NLP prompts across both stacks.
3. Compare: latency, quality, cost, and engineering effort.
4. Bonus: enforce JSON schema output and measure failure rate.

## Summary
- Proprietary serving is fastest to adopt.
- Hugging Face chat templates are essential for correct instruct-model prompting.
- vLLM gives strong throughput/latency for self-hosted inference when tuned with the right arguments.
- Embedding services need their own reliability and caching strategy, not just copied LLM settings.